# velofate - full workflow

The complete analysis behind the ccRCC single-cell atlas: from per-sample count matrices to RNA velocity, trajectories and cell-fate probabilities.

## Using this notebook with your own data

- **Input:** one `.h5ad` file per sample produced by the SLURM script (`kb count --workflow nac`), plus the `t2g.txt` file from `kb ref`. The `nac` workflow is required because it creates the `nascent`, `mature` and `ambiguous` layers that RNA velocity needs.
- **Sample names:** name each sample `<patient>_<T or N>`, e.g. `P2_T` and `P2_N` (several tumours: `P8_T1`, `P8_T2`). Patient and tumour/normal status are read from these names.
- **Kernel:** run the notebook with the `velofate` environment, from top to bottom.
- **Markers in the code** (search for them with Ctrl+F):
  - `# >>> CHANGE FOR YOUR DATA:` a value specific to this ccRCC dataset (file paths, sample names, marker genes, cluster numbers). It must be changed.
  - `# >>> TUNE:` a parameter that worked here. Adjust it if your results look off.
- **Cluster and state numbers do not transfer.** Leiden cluster numbers (`"0"`, `"4"`, `"7"`), CellRank macrostate names (`"2"`, `"1_1"`, `"3_2"`) and cell indices will be different in your data. Wherever the notebook selects clusters or states by number, look at the plots just above and choose your own.
- Plot titles mention ccRCC; they are only labels.

## Structure

- **Part A - whole dataset:** I. Preprocessing → II. Embedding → III. Integration & clustering → IV. Cell-type annotation. Always run this first.
- **Part B - compartments:** B1 epithelial (proximal tubule → ccRCC), B2 endothelial, B3 fibroblast/CAF, B4 myeloid (monocyte → macrophage). Each repeats the same steps on its own subset: subset and re-cluster → RNA velocity → pseudotime → CellRank fate → lineage drivers. Run only the compartments you need.

In [1]:
# Libraries used in this notebook
import scanpy as sc                  # preprocessing, clustering, UMAP, plotting
import scvelo as scv                 # RNA velocity
import bbknn                         # batch correction across patients

import cellrank as cr                # cell-fate analysis
from cellrank.kernels import VelocityKernel, ConnectivityKernel
from cellrank.estimators import GPCCA

import numpy as np
import pandas as pd
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text   # non-overlapping gene labels in driver plots

# Part A - Whole dataset

## I. Preprocessing
Load every sample, add metadata, convert gene IDs to gene names, remove low-quality cells and doublets, then normalise.

In [ ]:
# Load each sample's count matrix (.h5ad from the SLURM script) and merge them into one object.
# Loading multiple samples and merging
# >>> CHANGE FOR YOUR DATA: folder containing your .h5ad files
filepath = "/storage/homefs/la24s062/master_thesis/data"

# >>> CHANGE FOR YOUR DATA: one entry per sample, named <patient>_<T or N>, pointing to its .h5ad file
samples = {
    "P2_T": f"{filepath}/SRR19331815.h5ad",
    "P2_N": f"{filepath}/SRR19331819.h5ad",
    "P3_T": f"{filepath}/SRR19331879.h5ad",
    "P3_N": f"{filepath}/SRR19331804.h5ad",
    "P4_T": f"{filepath}/SRR19331807.h5ad",
    "P4_N": f"{filepath}/SRR19331811.h5ad",
    "P5_T": f"{filepath}/SRR19331859.h5ad",
    "P5_N": f"{filepath}/SRR19331875.h5ad",
    "P6_T": f"{filepath}/SRR19331897.h5ad",
    "P6_N": f"{filepath}/SRR19331901.h5ad",
    "P7_T": f"{filepath}/SRR19331847.h5ad", # Metastatic site: bone 
    "P7_N": f"{filepath}/SRR19331843.h5ad",
    "P8_T1": f"{filepath}/SRR19331840.h5ad", # Metastatic sites: bone, lung, adrenal gland, kidney
    "P8_T2": f"{filepath}/SRR19331836.h5ad", # Metastatic sites: bone, lung, adrenal gland, kidney   
    "P8_N": f"{filepath}/SRR19331832.h5ad",
}

# >>> CHANGE FOR YOUR DATA: per-sample annotation stored in the 'tumor_grade' column and used to colour plots.
#     Replace with your own (e.g. grade, stage, treatment); samples left out get NaN.
# Tumor grade annotation per sample
tumor_grade = {
    "P2_T": "3",
    "P3_T": "2",
    "P4_T": "2 . 3",
    "P5_T": "2 . 3",
    "P6_T": "3",
    "P7_T": "4", # Metastatic site: bone 
    "P8_T1": "4", # Metastatic sites: bone, lung, adrenal gland, kidney
    "P8_T2": "4", # Metastatic sites: bone, lung, adrenal gland, kidney   
}

adatas = []

for name, path in samples.items():
    adata = sc.read_h5ad(path) 
    adata.var_names_make_unique()  #Ensuring unique gene names

    #Annotating the sample, patient, condition and tumor grade information in the AnnData object
    suffix = name.split("_")[1]
    condition = "tumor" if suffix.startswith("T") else "normal"

    adata.obs["sample"]           = name
    adata.obs["patient"]          = name.split("_")[0]
    adata.obs["condition"]        = condition
    adata.obs["tumor_grade"]      = tumor_grade.get(name, np.nan)
    adatas.append(adata)


#Concatenating the individual AnnData objects into a single object
adata = sc.concat(
    adatas, 
    join="inner",               #Keeps only genes that are present in all samples
    merge="same",               #Merges the metadata columns with the same name
    uns_merge="same",           #Merges the unstructured annotation with the same name
    keys=list(samples.keys()),  #Adds a new column to the metadata with the sample names as values
    index_unique="-",           #Ensures unique cell barcodes by adding a suffix ("-P2_T")
)

adata

In [ ]:
# Convert Ensembl gene IDs to gene names (e.g. ENSG00000141510 -> TP53) using t2g.txt from kb ref.
# Load the t2g file
# >>> CHANGE FOR YOUR DATA: path to your t2g.txt
t2g = pd.read_csv("/storage/homefs/la24s062/master_thesis/data/t2g.txt", sep="\t", header=None)
t2g.columns = [
    "transcript_id", "gene_id", "gene_name", "transcript_name", 
    "chr", "start", "end", "strand"
]

# A. Clean IDs: Strip version numbers (ENSG000.1 -> ENSG000)
# We do this for both the t2g file and the adata object to ensure a match
t2g['gene_id_clean'] = t2g['gene_id'].str.split('.').str[0]
adata.var_names = adata.var_names.str.split('.').str[0]

# B. Create the mapping dictionary (one entry per gene_id)
gene_map = t2g.drop_duplicates('gene_id_clean').set_index('gene_id_clean')['gene_name'].to_dict()

# C. Store the original Ensembl IDs and map the new symbols
adata.var['ensembl_id'] = adata.var_names
adata.var['gene_symbol'] = adata.var_names.map(gene_map)

# D. Fill missing symbols with Ensembl IDs (safety check)
adata.var['gene_symbol'] = adata.var['gene_symbol'].fillna(adata.var['ensembl_id'])

# E. Set var_names to Gene Symbols and make them unique (e.g., MATR3 and MATR3-1)
adata.var_names = adata.var['gene_symbol']
adata.var_names_make_unique()

# F. Final check
print(f"Successfully created atlas with {adata.n_obs} cells and {adata.n_vars} genes.")
print(adata.obs.head())
print(adata.var.head())

In [ ]:
# Quality-control metrics per cell, plotted to choose the filtering thresholds in the next cell.
#Compute QC metrics
# >>> CHANGE FOR YOUR DATA: gene prefixes are for human; for mouse use "mt-", "Rps", "Rpl"
adata.var["mt"] = adata.var_names.str.startswith("MT-")               #Identifying mitochondrial genes based on gene names starting with "MT-"
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))    #Identifying ribosomal genes based on gene names starting with "RPS" or "RPL"

sc.pp.calculate_qc_metrics(
    adata, 
    qc_vars=["mt", "ribo"], 
    percent_top=None, 
    log1p=False, 
    inplace=True
)

print("Before filtering:", adata.n_obs)
print(adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe())

#Visualising QC metrics globally
sc.pl.violin(
    adata, 
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"], 
    jitter=0.4, 
    multi_panel=True
)

#Visualising QC metrics per sample
sc.pl.violin(
    adata, 
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample", 
    jitter=0.4, 
    multi_panel=True
)

# Scatter plots to inspect relationships between metrics
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

In [ ]:
# Remove low-quality cells and genes detected in fewer than 3 cells.
#Filtering low quality cells
# >>> CHANGE FOR YOUR DATA: thresholds chosen from the QC plots above for this kidney dataset;
#     set your own (the mitochondrial % cut-off in particular depends on the tissue)
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.n_genes_by_counts < 6000, :]
adata = adata[adata.obs.total_counts < 40000, :]
adata = adata[adata.obs.pct_counts_mt < 25, :]
adata = adata.copy()

sc.pp.filter_genes(adata, min_cells=3)

print("After filtering:", adata.n_obs)
print(adata.obs.groupby("sample").size())    

In [ ]:
# Detect and remove doublets (droplets containing two cells) with Scrublet, per sample.
# Doublets: run on filtered RAW counts, BEFORE normalization, per library.
# batch_key makes the simulated-doublet null per-sample, which is the correct null.
sc.pp.scrublet(adata, batch_key="sample")

# inspect before you cut - doublet rate should scale with cells loaded (~0.8%/1k)
print(adata.obs.groupby("sample")["predicted_doublet"].mean().round(3))
print("flagged doublets:", adata.obs["predicted_doublet"].sum())

# remove the confident few, but KEEP the score for post-clustering review
adata = adata[~adata.obs["predicted_doublet"]].copy()
print("After scrublet:", adata.n_obs)

In [7]:
# Make cells comparable: scale each cell to 10,000 counts, then log-transform.
#Normalisation and log transformation
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Save raw counts after normalising
adata.raw = adata  

In [ ]:
# Select the 2,000 most variable genes (per patient); these drive PCA, clustering and the UMAP.
#Highly variable gene selection
sc.pp.highly_variable_genes(
    adata, 
    flavor="seurat", 
    n_top_genes=2000,  
    batch_key="patient"
)

#Visualising highly variable genes
sc.pl.highly_variable_genes(adata)

#adata = adata[:, adata.var.highly_variable]

In [9]:
# Scale each gene so that highly expressed genes do not dominate the PCA.
#Scale the data
sc.pp.scale(adata, max_value=10, zero_center=False)

## II. Embedding
Summarise the variable genes into 50 main axes of variation (PCA).

In [ ]:
#PCA on highly variable genes
sc.tl.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack", random_state=27)

#Visualising PCA results
sc.pl.pca_variance_ratio(adata, log=True)

#Visualising PCA results
sc.pl.pca(adata, color=["sample", "patient", "condition", "tumor_grade"])

## III. Integration & clustering
Correct for differences between patients (BBKNN), build the UMAP and group cells into clusters (Leiden).

In [ ]:
import sklearn
import bbknn

#Batch correction with BBKNN
bbknn.bbknn(
    adata,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

#UMAP
sc.tl.umap(adata, random_state=27)

#Cluster the data using Leiden algorithm
# >>> TUNE: clustering resolution (higher = more, smaller clusters)
sc.tl.leiden(adata, resolution=0.5, random_state=27)

#Visualising UMAP results
sc.pl.umap(adata, color=["patient", "condition", "sample", "leiden", "tumor_grade"])

In [12]:
# Optional (disabled): automatic immune cell-type annotation with CellTypist. Uncomment to use.
#import celltypist
#from celltypist import models

#pred = celltypist.annotate(adata, model="Immune_All_Low.pkl", majority_voting=True)

#adata.obs["celltypist"] = pred.predicted_labels["majority_voting"].values

#sc.pl.umap(adata, color="celltypist")

## IV. Cell-type annotation
Score every cluster for known marker genes and label it with the best-matching cell type.

In [ ]:
# Marker genes per cell type, shown as a dot plot across clusters.
# >>> CHANGE FOR YOUR DATA: kidney / ccRCC marker sets; replace with markers for your tissue.
#     Later sections select cells by these names (e.g. "Proximal tubule", "Endothelial", "Monocytes"),
#     so update those selections too if you rename them.
markers = {
    # --- MALIGNANT & EPITHELIAL ---
    "Tumor cells (ccRCC)": ["CA9", "NDUFA4L2", "VEGFA", "PLIN2", "NNMT", "EGLN3", "SLC2A1"],
    "Proximal tubule": ["MIOX", "GATM", "CUBN", "LRP2"],
    "Distal/Collecting duct": ["CA12", "ATP6V1B1", "CLCNKB", "DEFB1", "TMEM213", "AQP2"],
    "Epithelial (General)": ["EPCAM", "KRT8", "KRT18", "PAX8"],

    # --- LYMPHOID IMMUNE ---
    "T cells (General)": ["CD3D", "CD3E", "CD3G"],
    "CD8 T cells": ["CD8A", "CD8B", "GZMB", "PRF1"],
    "CD4 T cells": ["CD4", "IL7R", "LTB"],
    "Tregs": ["FOXP3", "IL2RA", "CTLA4"],
    "Exhausted T cells": ["PDCD1", "LAG3", "TIGIT", "HAVCR2"], # HAVCR2 = TIM3
    "NK cells": ["NCAM1", "KLRD1", "GNLY", "PRF1"],
    "B cells": ["CD79A", "MS4A1", "CD19"],
    "Plasma cells": ["MZB1", "SDC1", "CD38", "JCHAIN"], # essential for the ccRCC TME

    # --- MYELOID IMMUNE ---
    "Monocytes": ["S100A8", "S100A9", "FCN1"], # distinguishes monocytes from macrophages
    "Macrophages (General)": ["LYZ", "CD68", "CD14", "APOE"],
    "Macrophages (M2-like/TAM)": ["CD163", "MRC1", "MARCO"], 
    "Dendritic cells": ["CLEC9A", "FCER1A", "CD1C"],
    "Mast cells": ["TPSAB1", "CPA3", "KIT"], # common in renal tissue

    # --- STROMAL/VASCULAR ---
    "Fibroblasts": ["COL1A1", "DCN", "FAP", "LUM"],
    "Myofibroblasts (CAFs)": ["ACTA2", "TAGLN", "POSTN"], # marks aggressive stroma
    "Endothelial": ["PECAM1", "VWF", "CDH5", "PLVAP"], # PLVAP is high in tumor vessels
    "Pericytes/Smooth Muscle": ["RGS5", "PDGFRB", "MCAM"], # stabilise tumour vessels

    # --- CELL STATE ---
    "Proliferation": ["MKI67", "TOP2A", "STMN1", "PCNA"]
}

# Dotplot for overview
sc.pl.dotplot(adata, markers, groupby="leiden", standard_scale="var")

In [ ]:
# Key marker genes on the UMAP.
# >>> CHANGE FOR YOUR DATA: genes to plot
sc.pl.umap(
    adata,
    color=[
        "CA9",        # tumor
        "NDUFA4L2",      # tumor (better than VEGFA)
        "CD3D",       # T cells
        "CD8A",       # CD8 T
        "FOXP3",      # Tregs
        "NKG7",       # NK
        "CD68",       # myeloid
        "PECAM1",     # endothelial
        "COL1A1",     # fibroblast
        "EPCAM",      # epithelial/tumor 
        "CLCNKB",     # Distal tubule/Collecting duct
        "CA12",       # Distal tubule/Collecting duct
        "MIOX",       # Proximal tubule
        "GATM",       # Proximal tubule
        "MME",         # CD10        
        "CSPG4"        #pericytes     
    ],
    cmap="viridis",
    ncols=4
)

In [ ]:
# Score every cell for each marker set and plot the scores on the UMAP.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

In [ ]:
# Label each cluster with its best-scoring cell type -> column 'cell_type1' (used for the endothelial subset).
# --- Cluster-level annotation from marker scores (replaces per-cell idxmax) ---

# 1. Keep cell-STATE signatures out of the identity competition.
#    Proliferation is a state, not a type - a cycling tumor cell is still a tumor cell.
state_signatures = ["Proliferation"]
identity_signatures = [k for k in markers if k not in state_signatures]

# 2. Average per-cell scores within each Leiden cluster (stable; kills per-cell noise)
cluster_scores = adata.obs.groupby("leiden", observed=True)[identity_signatures].mean()

# 3. Z-score each signature ACROSS clusters so different gene sets become comparable
cluster_z = (cluster_scores - cluster_scores.mean()) / cluster_scores.std()

# 4. Assign each cluster its top signature; flag low-confidence calls as 'Ambiguous'
cluster_labels = cluster_z.idxmax(axis=1)
top2 = np.sort(cluster_z.values, axis=1)[:, -2:]
margin = pd.Series(top2[:, 1] - top2[:, 0], index=cluster_z.index)
# >>> TUNE: minimum score gap between the best and second-best cell type
cluster_labels[margin < 0.01] = "Ambiguous"   # tune AFTER looking at the heatmap below

# 5. Map cluster labels back to cells
# Start from the automated calls, then override where the heatmap is clearly right
# >>> CHANGE FOR YOUR DATA: manual label corrections for specific cluster numbers; decide yours from the heatmap
manual = {
    "1": "Exhausted T-cells (general)",
    "5": "Fibroblasts / CAFs / Pericytes",   # co-fires Fibroblast/CAF/Pericyte -> stromal; resolved finely in the stroma subset
    "11": "Tumor cells (ccRCC) / Proximal tubule",
}
cluster_labels.update(pd.Series(manual))
adata.obs["cell_type1"] = adata.obs["leiden"].map(cluster_labels).astype("category")

# 6. Validate
import seaborn as sns
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_z, cmap="RdBu_r", center=0)
plt.title("Cluster x signature (z-scored mean) - confirm each cluster's top call is sensible")
plt.show()
print(cluster_labels.to_string())

sc.pl.umap(adata, color=["leiden", "cell_type1"], legend_loc="on data",
           legend_fontsize=6, legend_fontoutline=1)

In [ ]:
# UMAP with the cluster-level cell-type labels.

sc.pl.umap(
    adata,
    color=["cell_type1"],
    legend_loc="on data",
    legend_fontsize=6,
    legend_fontoutline=0.7
)

In [ ]:
# UMAP coloured by patient: check that patients mix well after integration.
sc.pl.umap(adata, color=["patient"], legend_loc="right margin")

In [ ]:
# Alternative per-cell labelling -> column 'cell_type' (used for the epithelial, fibroblast and myeloid subsets).
#Assigning cell type labels based on the highest scoring gene set
score_cols = list(markers.keys())

adata.obs["cell_type"] = (
    adata.obs[score_cols]
    .idxmax(axis=1)
)

sc.pl.umap(
    adata,
    color=["cell_type"],
    legend_loc="on data",
    legend_fontsize=6,
    legend_fontoutline=1
)

# Part B - Compartment analyses
Each major compartment is taken out of the full dataset, re-clustered on its own and analysed for RNA velocity, trajectories and cell fate.

## B1. Epithelial compartment: proximal tubule → ccRCC
### Subset and re-cluster

In [ ]:
# Keep only epithelial cells, then redo variable genes, PCA, integration, UMAP and clustering for this subset.
# Subset
# >>> CHANGE FOR YOUR DATA: cell-type label(s) (names from `markers`) that make up this compartment
adata_tumor = adata[adata.obs["cell_type"].isin(["Proximal tubule", "Tumor cells (ccRCC)"])].copy()

# Re-calculate Features (Specific to this lineage)
sc.pp.highly_variable_genes(adata_tumor, n_top_genes=2000, batch_key='patient')

# Re-calculate Embedding
sc.tl.pca(adata_tumor, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

bbknn.bbknn(
    adata_tumor,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

sc.tl.umap(adata_tumor, random_state=27)

# >>> TUNE: clustering resolution
sc.tl.leiden(adata_tumor, resolution=0.4, key_added="tumor_subclusters", random_state=27)

sc.pl.umap(adata_tumor, color=["condition", "patient", "tumor_grade", "tumor_subclusters"])


In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_tumor, 'tumor_subclusters', method='wilcoxon')
sc.pl.rank_genes_groups(adata_tumor, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_tumor.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

In [ ]:
# Marker-set scores on the subset UMAP, used to spot contaminating cell types.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata_tumor, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata_tumor,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

In [ ]:
# Remove contaminating sub-clusters and re-embed the cleaned subset.
# >>> CHANGE FOR YOUR DATA: sub-cluster numbers to keep; choose them from the plots above
adata_tumor_clean = adata_tumor[adata_tumor.obs['tumor_subclusters'].isin(["0", "4", "7"])].copy()

# 1. HVGs: Find genes that vary ONLY along the PT -> ccRCC axis
# This is where you find the true drivers of malignancy
sc.pp.highly_variable_genes(adata_tumor_clean, n_top_genes=2000, batch_key='patient')

# 2. PCA: Define the variation space for just this lineage
sc.tl.pca(adata_tumor_clean, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

# 3. BBKNN: Final integration to ensure patient-specific effects are gone
bbknn.bbknn(
    adata_tumor_clean,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

# 4. UMAP: This should now show a beautiful, continuous trajectory
sc.tl.umap(adata_tumor_clean, random_state=27)
sc.pl.umap(adata_tumor_clean, color=["condition", "patient", "tumor_grade", "tumor_subclusters"])


In [ ]:
# Run Leiden clustering on the refined BBKNN graph
# key_added prevents overwriting your previous 'tumor_subclusters'
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_tumor_clean, resolution=0.3, key_added='tumor_fine_grain')

# Visualize the new clusters
sc.pl.umap(adata_tumor_clean, color=['tumor_fine_grain', 'patient', 'tumor_grade'])

In [ ]:
# Selected genes on the UMAP.
# >>> CHANGE FOR YOUR DATA: genes to plot
sc.pl.umap(
    adata_tumor_clean,
    color=["MME", "PAX8", "FTL", "FTH1", "GPX4", "STAT1", "EPAS1", "CD163", "MRC1", "CP", "MET"],
    cmap="viridis"
)

In [ ]:
# Identify marker genes for the new fine-grained clusters
sc.tl.rank_genes_groups(adata_tumor_clean, 'tumor_fine_grain', method='wilcoxon')

# Plot the top 5 genes for each new cluster to see the differences
sc.pl.rank_genes_groups_dotplot(adata_tumor_clean, n_genes=5, values_to_plot='logfoldchanges')

In [ ]:
# Same test, showing the top 50 genes per sub-cluster.
sc.tl.rank_genes_groups(adata_tumor_clean, 'tumor_fine_grain', method='wilcoxon')
sc.pl.rank_genes_groups(adata_tumor_clean, n_genes=50, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_tumor_clean.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

### Iron and ferroptosis genes (project-specific)
Optional check specific to this study; skip it or replace the gene lists.

In [ ]:
# Mean expression, % of expressing cells and z-score per cluster for iron / ferroptosis genes.
import scanpy as sc, pandas as pd, numpy as np

# >>> CHANGE FOR YOUR DATA: gene sets of interest (specific to this study)
iron = ['FTL','FTH1','SLC40A1','TFRC','SLC11A2','STEAP3','ACO1','IREB2',
        'HMOX1','FTMT','SLC39A14','HEPH']
ferro = ['GPX4', "NSD2", 'SLC7A11','SLC3A2','ACSL4','LPCAT3','AIFM2','PTGS2','NFE2L2','NCOA4']
genes = iron + ferro

key = 'tumor_fine_grain'

# sanity check on what X holds
print(adata_tumor_clean.X.min(), adata_tumor_clean.X.max(), adata_tumor_clean.layers.keys())

present = [g for g in genes if g in adata_tumor_clean.var_names]
print('missing:', set(genes) - set(present))

df = sc.get.obs_df(adata_tumor_clean, keys=present + [key])

mean_expr = df.groupby(key).mean()                       # raw mean per cluster
pct_expr  = df.groupby(key).apply(lambda d: (d[present] > 0).mean() * 100)
zscore    = (mean_expr - mean_expr.mean()) / mean_expr.std()   # across clusters

pd.set_option('display.width', 200, 'display.max_columns', 50)
print(mean_expr.round(2))
print(pct_expr.round(1))
print(zscore.round(2))

In [ ]:
# Dot plot of the same genes.
sc.pl.dotplot(adata_tumor_clean, {'iron': [g for g in iron if g in present],
                                  'ferroptosis': [g for g in ferro if g in present]},
              groupby=key, standard_scale='var', cmap='viridis')

### RNA velocity
RNA velocity compares newly made (unspliced) with mature (spliced) RNA in each cell to estimate the direction in which cells are changing. Arrows on the UMAP point toward each cell's likely future state.

In [31]:
# scVelo display settings.
scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.presenter_view = True  # set max width size for presenter view
scv.set_figure_params('scvelo')  # for beautified visualization

In [ ]:
# Rename the kb-python layers for scVelo: nascent -> unspliced, mature + ambiguous -> spliced.
# (Requires .h5ad files made with `kb count --workflow nac`, as in the SLURM script.)
# Map Nascent to Unspliced
adata_tumor_clean.layers['unspliced'] = adata_tumor_clean.layers['nascent'] # Nascent reads definitively contain intronic sequences (unspliced)

# Map Mature (+ Ambiguous) to Spliced. Adding ambiguous to spliced to recover reads that overlap both an exon and a boundary.
adata_tumor_clean.layers['spliced'] = adata_tumor_clean.layers['mature'] + adata_tumor_clean.layers['ambiguous'] # Mature reads are purely exonic (spliced)

# Remove the old keys so scVelo doesn't get confused
del adata_tumor_clean.layers['nascent']
del adata_tumor_clean.layers['mature']
del adata_tumor_clean.layers['ambiguous']

# Verify the proportions
scv.utils.show_proportions(adata_tumor_clean)
scv.pl.proportions(adata_tumor_clean)

In [ ]:
# --- 2. Velocity-Specific Preprocessing ---
# This filters specifically for genes with sufficient spliced/unspliced counts
# >>> TUNE: minimum spliced and unspliced counts a gene needs to be kept
scv.pp.filter_genes(adata_tumor_clean, min_shared_counts=20)

# If your adata is already normalized/log-transformed, scVelo can 
# usually detect this. But we run this to ensure the layers are handled correctly.
scv.pp.normalize_per_cell(adata_tumor_clean)

In [ ]:
# Subset inherited SCALED .X from the scaling step in Part A, which starved HVG selection (-> only 10 velocity genes).
# Restore log-normalized expression from .raw and re-flag HVGs so velocity gets a full candidate pool.
adata_tumor_clean.X = adata_tumor_clean.raw.to_adata()[:, adata_tumor_clean.var_names].X.copy()
sc.pp.highly_variable_genes(adata_tumor_clean, flavor="seurat", n_top_genes=2000, batch_key="patient")
print("HVGs after fix:", int(adata_tumor_clean.var["highly_variable"].sum()))

In [ ]:
# --- 3. The "BBKNN Preservation" Step ---
# DO NOT run sc.pp.neighbors or sc.pp.pca here. 
# We want to use the integrated graph created by BBKNN earlier.

print("Computing moments using the BBKNN-integrated graph...")
scv.pp.moments(adata_tumor_clean, n_pcs=None, n_neighbors=None)

In [ ]:
import tqdm
import ipywidgets
# --- 4. Compute Velocity ---
# Now calculate the actual velocity vectors
scv.tl.velocity(adata_tumor_clean, mode="stochastic") # 'stochastic' is faster/stable for first runs
scv.tl.velocity_graph(adata_tumor_clean, n_jobs=8)

In [ ]:
# Project velocities onto a UMAP embedding
scv.tl.velocity_embedding(adata_tumor_clean, basis='umap')

In [ ]:
# Velocity streamlines on the UMAP.
scv.pl.velocity_embedding_stream(
    adata_tumor_clean, 
    basis='umap', 
    color=["tumor_fine_grain", "tumor_subclusters", "patient"],
    legend_loc='right margin', 
    title="RNA Velocity of ccRCC Atlas",
    smooth=0.6  # Adjust this to make arrows more or less "curvy"
)

### Figure labels

In [39]:
# Readable names for the main sub-clusters.
import pandas as pd

# >>> CHANGE FOR YOUR DATA: sub-cluster number -> name
cluster_labels = {
    "0": "1: Main tumor state",
    "4": "2: CP-high terminal tumor state",
    "7": "6: Healthy proximal tubule",
}

adata_tumor_clean.obs["tumor_subclusters_labelled"] = (
    adata_tumor_clean.obs["tumor_subclusters"]
    .astype(str)
    .replace(cluster_labels)
    .astype("category")
)

In [ ]:
# A fixed colour for each named sub-cluster.
new_key = "tumor_subclusters_labelled"

# original cluster ID -> colour
# >>> CHANGE FOR YOUR DATA: one colour per sub-cluster number named above
cluster_colors = {
    "0": "#F28E2B",   # orange
    "4": "#32B44A",   # green
    "7": "#e377c2",   # pink
}

# label -> colour, via the cluster_labels dict from the previous cell
label_to_color = {label: cluster_colors[cid] for cid, label in cluster_labels.items()}

adata_tumor_clean.uns[f"{new_key}_colors"] = [
    label_to_color[cat] for cat in adata_tumor_clean.obs[new_key].cat.categories
]

sc.pl.umap(adata_tumor_clean, color=new_key)

In [ ]:
# Velocity streamlines with the named sub-clusters (figure version).
scv.pl.velocity_embedding_stream(
    adata_tumor_clean,
    basis="umap",
    color="tumor_subclusters_labelled",
    legend_loc="right margin",
    title="RNA Velocity of ccRCC Atlas",
    smooth=0.9,
    size=450,
    alpha=0.6
)

### Per-patient checks

In [ ]:
# Cells per patient in each sub-cluster, used to pick a well-balanced patient for the check below.
ct = pd.crosstab(adata_tumor_clean.obs["patient"], adata_tumor_clean.obs["tumor_subclusters"])
print(ct)                                   # rows = patient, cols = subcluster (7 = PT, 0 = tumor)

# fraction of PT cells per patient, and absolute PT count
# >>> CHANGE FOR YOUR DATA: "7" = healthy proximal tubule and "0" = tumour sub-cluster in this dataset
pt_frac = (ct.get("7", 0) / ct.sum(axis=1)).round(3)
summary = pd.DataFrame({"PT_count": ct.get("7", 0), "tumor_count": ct.get("0", 0), "PT_frac": pt_frac})
print(summary.sort_values("PT_count", ascending=False))

In [ ]:
# Expression of one gene within one sub-cluster, broken down by patient.
import numpy as np
import pandas as pd
import scipy.sparse as sp

# >>> CHANGE FOR YOUR DATA: gene and sub-cluster to examine
GENE    = "CP"
CLUSTER = "4"

ct = pd.crosstab(adata_tumor_clean.obs["patient"],
                 adata_tumor_clean.obs["tumor_subclusters"])
print(ct)

# >>> CHANGE FOR YOUR DATA: "7" = proximal tubule and "0" = tumour sub-cluster (as in the previous cell)
pt_frac = (ct.get("7", 0) / ct.sum(axis=1)).round(3)
summary = pd.DataFrame({"PT_count": ct.get("7", 0),
                        "tumor_count": ct.get("0", 0),
                        "PT_frac": pt_frac})
print(summary.sort_values("PT_count", ascending=False))

# ── gene expression within one cluster, broken down by patient ──────
src = adata_tumor_clean.raw if adata_tumor_clean.raw is not None else adata_tumor_clean
if GENE not in src.var_names:
    raise ValueError(f"{GENE} not in var_names")

mask = (adata_tumor_clean.obs["tumor_subclusters"] == CLUSTER).values
sub  = src[mask, GENE] if src is adata_tumor_clean.raw else adata_tumor_clean[mask, GENE]

expr = sub.X
expr = expr.toarray().ravel() if sp.issparse(expr) else np.asarray(expr).ravel()

df = pd.DataFrame({
    "patient": adata_tumor_clean.obs.loc[mask, "patient"].values,
    "expr": expr,
})

res = df.groupby("patient", observed=True).agg(
    n_cells      = ("expr", "size"),
    n_pos        = ("expr", lambda x: int((x > 0).sum())),
    mean_expr    = ("expr", "mean"),
    mean_in_pos  = ("expr", lambda x: x[x > 0].mean() if (x > 0).any() else 0.0),
)
res["pct_pos"] = (100 * res["n_pos"] / res["n_cells"]).round(1)
res = res[["n_cells", "n_pos", "pct_pos", "mean_expr", "mean_in_pos"]].round(3)

print(f"\n{GENE} in cluster {CLUSTER}, by patient:")
print(res.sort_values("pct_pos", ascending=False).to_string())

In [ ]:
# The same gene per patient as violin and dot plot.
sc.pl.violin(adata_tumor_clean[mask], GENE, groupby="patient", rotation=90, use_raw=True)
sc.pl.dotplot(adata_tumor_clean[mask], [GENE], groupby="patient", use_raw=True)

In [ ]:
# Sensitivity check: velocity recomputed within one patient without integration, to confirm the
# direction is not an artefact of BBKNN.
# >>> CHANGE FOR YOUR DATA: patient ID
pt = "P3"   # <-- replace with the best-balanced patient from the table above
sub = adata_tumor_clean[adata_tumor_clean.obs["patient"] == pt].copy()

sc.pp.highly_variable_genes(sub, n_top_genes=2000)
sc.tl.pca(sub, n_comps=30, svd_solver="arpack", random_state=27)
sc.pp.neighbors(sub, n_pcs=30, n_neighbors=15, random_state=27)
sc.tl.umap(sub, random_state=27)

scv.pp.moments(sub, n_pcs=None, n_neighbors=None)
scv.tl.velocity(sub, mode="stochastic")
scv.tl.velocity_graph(sub, n_jobs=1)
scv.pl.velocity_embedding_stream(sub, basis="umap", color="tumor_subclusters",
                                 title=f"Velocity within {pt} (no integration)")

*Note:* velocity was computed on the patient-integrated (BBKNN) graph. To rule out integration artefacts, it was recomputed within a single patient without integration (cell above); the inferred direction was concordant.

In [46]:
# Optional (disabled): grid-style velocity plot.
# The Grid plot is often better for seeing general trends in large atlases
#scv.pl.velocity_embedding_grid(adata, basis='umap', color='cell_type', arrow_size=2)

In [ ]:
# Velocity confidence per cell, and the genes that drive velocity in each sub-cluster.
# Identify genes that are "speeding up" as cells become malignant - genes that are actively being transcribed at the moment of transition
scv.tl.velocity_confidence(adata_tumor_clean)

scv.tl.rank_velocity_genes(adata_tumor_clean, groupby="tumor_fine_grain")

In [ ]:
# Velocity confidence and speed per cell on the UMAP.
scv.pl.scatter(adata_tumor_clean, 
               color=['velocity_confidence', 'velocity_length'], 
               cmap='coolwarm', 
               perc=[5, 95])

In [ ]:
# Table of the top 10 velocity genes per sub-cluster.
# Extract the ranked names into a DataFrame
top_genes = pd.DataFrame(adata_tumor_clean.uns["rank_velocity_genes"]["names"])
top_genes.head(10)

In [ ]:
# Phase portraits: unspliced vs spliced RNA for selected genes.
# List the genes from your output
# >>> CHANGE FOR YOUR DATA: genes to show (pick from the velocity-gene table above)
target_genes = ["ACSM2B", "CP", "SERPINE1", "ENPP1"]
# Use scv.pl.scatter to generate phase portraits
# 'color' allows you to see where each cluster sits on the curve
scv.pl.scatter(adata_tumor_clean, 
               basis=target_genes, 
               color=['tumor_fine_grain'],
               frameon=False,
               ncols=2)

In [ ]:
# Project the velocity arrows for these specific genes onto the UMAP
scv.pl.velocity(adata_tumor_clean, 
                var_names=target_genes, 
                color='tumor_fine_grain')

In [ ]:
# Velocity graph: which cells are predicted to transition into which.
scv.pl.velocity_graph(adata_tumor_clean, color='tumor_subclusters', threshold=.1)

In [ ]:
# Velocity pseudotime: each cell's position along the inferred trajectory (0 = start, 1 = end).

scv.tl.velocity_pseudotime(adata_tumor_clean)
scv.pl.scatter(adata_tumor_clean, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
# Follow the predicted path of a single starting cell across the UMAP.
# >>> CHANGE FOR YOUR DATA: starting_cell is a cell index (row number); pick a cell in your starting population
x, y = scv.utils.get_cell_transitions(adata_tumor_clean, basis='umap', starting_cell=65)
ax = scv.pl.velocity_graph(adata_tumor_clean, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata_tumor_clean, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

### Trajectory inference
PAGA summarises how strongly the sub-clusters are connected, giving a simplified map of the trajectory.

In [ ]:
# PAGA: simplified map of how strongly the sub-clusters are connected.
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

# 1. Compute PAGA connectivity between sub-clusters
sc.tl.paga(adata_tumor_clean, groups='tumor_subclusters')

# 2. Plot it (this shows the 'skeleton' of your data)
scv.pl.paga(adata_tumor_clean, basis='umap', size=50, alpha=.1, 
            min_edge_width=2, node_size_scale=1.5)

### Cell fate analysis
Diffusion pseudotime from a chosen root cell, then CellRank: it identifies the start and end states of the trajectory and the probability that each cell reaches each end state.

In [ ]:
# Set root in cluster 7 using GATM (a proximal tubule marker)
# >>> CHANGE FOR YOUR DATA: starting cluster ("7") and a marker gene of it ("GATM");
#     the root is the cell in that cluster with the highest expression of the gene
cluster7_mask = adata_tumor_clean.obs["tumor_subclusters"] == "7"
gatm_expr = adata_tumor_clean[cluster7_mask, "GATM"].X
if hasattr(gatm_expr, "toarray"):
    gatm_expr = gatm_expr.toarray().flatten()
adata_tumor_clean.uns["iroot"] = np.where(cluster7_mask)[0][np.argmax(gatm_expr)]

# Rerun diffmap and DPT
sc.tl.diffmap(adata_tumor_clean, n_comps=15)
sc.tl.dpt(adata_tumor_clean, n_dcs=10)

# Visualize
sc.pl.umap(adata_tumor_clean, color=["dpt_pseudotime"], cmap="gnuplot")

In [ ]:
# CellRank transition model (80% RNA velocity + 20% transcriptional similarity) and its spectrum.
import cellrank as cr
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel, VelocityKernel
from cellrank.estimators import GPCCA

# Initialize the kernels
vk = VelocityKernel(adata_tumor_clean).compute_transition_matrix()
ck = ConnectivityKernel(adata_tumor_clean).compute_transition_matrix()

# Combine them (80% velocity, 20% transcriptomic similarity)
# >>> TUNE: weighting between velocity and similarity
combined_kernel = 0.8 * vk + 0.2 * ck

# Rerun estimator
g = GPCCA(combined_kernel)
# >>> TUNE: number of components to compute
g.compute_schur(n_components=10)
g.plot_spectrum(real_only=True)

In [ ]:
# Compute macrostates - coarse grained states of the trajectory
# >>> TUNE: number of macrostates; a gap in the spectrum plot above suggests a good value
g.compute_macrostates(n_states=8, cluster_key="tumor_fine_grain")
g.plot_macrostates(which="all", discrete=False)

# Check how many cells are in each macrostate
print(g.macrostates.value_counts())

# Check macrostate memberships - should show soft assignments for all cells
print(g.macrostates_memberships)

# Plot memberships instead of hard assignments
g.plot_macrostates(which="all", discrete=False)  # continuous membership

In [ ]:
# Macrostate names (needed to set the start and end states below).
print(g.macrostates.cat.categories)

In [ ]:
import cellrank as cr

# --- Unsupervised check: let CellRank find terminal states from the graph itself ---
g_unsup = cr.estimators.GPCCA(combined_kernel)   # same kernel you already built

# 1. Compute macrostates. Ask for a few more than you expect (PT + tumor states + a couple).
g_unsup.compute_schur(n_components=20)
# >>> TUNE: number of macrostates
g_unsup.compute_macrostates(n_states=6, cluster_key="tumor_fine_grain")

# 2. Let CellRank DECIDE which macrostates are terminal (no manual input)
g_unsup.predict_terminal_states()          # data-driven, based on coarse-grained stationarity
g_unsup.predict_initial_states()           # likewise for the root

# 3. Inspect what it chose
print("Predicted terminal states:")
print(g_unsup.terminal_states)             # which cells/clusters it flagged as sinks
print("Predicted initial states:")
print(g_unsup.initial_states)

g_unsup.plot_macrostates(which="terminal", basis="umap", legend_loc="right")
g_unsup.plot_macrostates(which="initial",  basis="umap", legend_loc="right")

In [ ]:
# Set start and end states, then compute each cell's probability of reaching each end state.
# 7. Set initial (start) and terminal (end) states
#g.predict_terminal_states()
# >>> CHANGE FOR YOUR DATA: macrostate names from the plots above (initial = start, terminal = end states)
g.set_initial_states(states=['6'])
g.set_terminal_states(states=["2", "3", "4"])

g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot=False)
# Then verify what was identified
#sc.pl.umap(adata, color=["macrostates_fwd", "terminal_states"])
#print("Terminal states identified:", adata.obs["terminal_states"].cat.categories.tolist())
#g.plot_macrostates(which="terminal", legend_loc="right")



In [ ]:
# Dynamical velocity model -> latent time, a second estimate of each cell's position along the trajectory.
scv.pp.moments(adata_tumor_clean)

# 2. Run the dynamical model
# This fits the transcriptional dynamics for each gene
scv.tl.recover_dynamics(adata_tumor_clean)

# 3. Compute velocity using the dynamical model
# 'mode' must be explicitly set to 'dynamical'
scv.tl.velocity(adata_tumor_clean, mode='dynamical')

# 4. Now you can compute latent time
scv.tl.latent_time(adata_tumor_clean)

# 5. Finally, plot it
import scanpy as sc
sc.pl.umap(adata_tumor_clean, color='latent_time', cmap='gnuplot')
scv.pl.velocity_embedding_stream(adata_tumor_clean, basis='umap', color='latent_time', title='Kinetic Age and Trajectory')

### Lineage drivers
Genes whose expression rises with the probability of reaching a given end state.

In [ ]:
# Lineage drivers: genes correlated with the probability of reaching each end state.
# 6. Find genes driving a specific lineage (e.g., "Malignant_Terminal_State")
lineage_drivers = g.compute_lineage_drivers(return_drivers=True)

# 7. View the top 20 drivers for your terminal state tumor cluster
# >>> CHANGE FOR YOUR DATA: "2" = terminal macrostate name
lineage_drivers.sort_values(by="2_corr", ascending=False).head(20)

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "2" = terminal macrostate name
g.plot_lineage_drivers(lineage="2", n_genes=20)

In [ ]:
# Volcano-style plot: correlation with fate probability vs significance; top 20 drivers labelled.
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# >>> CHANGE FOR YOUR DATA: "2" = terminal macrostate name
lin = "2"
df = adata_tumor_clean.varm["terminal_lineage_drivers"].copy()
df = df[[f"{lin}_corr", f"{lin}_qval"]].dropna()
df.columns = ["corr", "qval"]
df["neglog_q"] = -np.log10(df["qval"].clip(lower=1e-300))

top20 = df[df["corr"] > 0].nlargest(20, "neglog_q")

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(df["corr"], df["neglog_q"], s=6, c="lightgrey", alpha=0.5, rasterized=True)
ax.scatter(top20["corr"], top20["neglog_q"], s=30, c="crimson", zorder=3)

ax.set_yscale("log")
# >>> TUNE: axis limits for this lineage (cosmetic)
ax.set_ylim(10, 400)
ax.set_xlim(0, 0.95)

texts = [ax.text(r["corr"], r["neglog_q"], g, fontsize=18, zorder=4)
         for g, r in top20.iterrows()]
adjust_text(texts, ax=ax,
            expand_text=(1.6, 1.8),
            force_text=(0.6, 1.2),
            arrowprops=dict(arrowstyle="-", lw=0.6, color="grey"))

ax.axvline(0, ls="--", lw=0.8, c="black")
ax.set_xlabel(f"Correlation with lineage {lin} fate probability", fontsize=18)
ax.set_ylabel("-log10(q-value)  [log scale]", fontsize=18)
ax.set_title(f"Lineage {lin} drivers", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 drivers for another end state.
# >>> CHANGE FOR YOUR DATA: "4" = terminal macrostate name
lineage_drivers.sort_values(by="4_corr", ascending=False).head(20)

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "4" = terminal macrostate name
g.plot_lineage_drivers(lineage="4", n_genes=20)

In [ ]:
# Top 20 drivers for another end state.
# >>> CHANGE FOR YOUR DATA: "3" = terminal macrostate name
lineage_drivers.sort_values(by="3_corr", ascending=False).head(20)

In [ ]:
# Volcano-style plot: correlation with fate probability vs significance; top 20 drivers labelled.
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# >>> CHANGE FOR YOUR DATA: "3" = terminal macrostate name
lin = "3"
df = adata_tumor_clean.varm["terminal_lineage_drivers"].copy()
df = df[[f"{lin}_corr", f"{lin}_qval"]].dropna()
df.columns = ["corr", "qval"]
df["neglog_q"] = -np.log10(df["qval"].clip(lower=1e-300))

top20 = df[df["corr"] > 0].nlargest(20, "neglog_q")

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(df["corr"], df["neglog_q"], s=6, c="lightgrey", alpha=0.5, rasterized=True)
ax.scatter(top20["corr"], top20["neglog_q"], s=30, c="crimson", zorder=3)

ax.set_yscale("log")
# >>> TUNE: axis limits for this lineage (cosmetic)
ax.set_ylim(1, 125)             # crop the empty upper region
ax.set_xlim(0, 0.4)        # crop the empty left tail; leaves room for the GRIA4 label

texts = [ax.text(r["corr"], r["neglog_q"], g, fontsize=18, zorder=4)
         for g, r in top20.iterrows()]
adjust_text(texts, ax=ax,
            expand_text=(1.6, 1.8),
            force_text=(0.6, 1.2),
            arrowprops=dict(arrowstyle="-", lw=0.6, color="grey"))

ax.axvline(0, ls="--", lw=0.8, c="black")
ax.set_xlabel(f"Correlation with lineage {lin} fate probability", fontsize=18)
ax.set_ylabel("-log10(q-value)  [log scale]", fontsize=18)
ax.set_title(f"Lineage {lin} drivers", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "3" = terminal macrostate name
g.plot_lineage_drivers(lineage="3", n_genes=20)

In [ ]:
# PAGA on the fine-grained clusters, with arrows.
sc.tl.paga(adata_tumor_clean, groups='tumor_fine_grain')

# 2. Plot PAGA with arrows (thresholding weak edges for clarity)
# This will show a "tube-like" flow from PT to Tumor
sc.pl.paga(adata_tumor_clean, threshold=0.1, edge_width_scale=0.5, 
           arrowsize=15, title="Directed PAGA: PT to ccRCC Progression")

## B2. Endothelial compartment (vascular niche)
Same steps as B1. Endothelial cells are taken from the cluster-level labels (`cell_type1`).
### Subset and re-cluster

In [ ]:
# 1. Perform the subsetting selection
# >>> CHANGE FOR YOUR DATA: cell-type label(s) (names from `markers`) that make up this compartment
selection = adata.obs["cell_type1"].isin([
    "Endothelial"
]).values

# 2. MANUALLY REBUILD the object to drop broken obsp/uns
# This takes the expression matrix (X), observations (obs), and genes (var) 
# and puts them into a brand new, clean container.
adata_endothelial = sc.AnnData(
    X=adata[selection].X.copy(),
    obs=adata[selection].obs.copy(),
    var=adata.var.copy(),
    layers={key: layer[selection].copy() for key, layer in adata.layers.items()}
)

adata_endothelial.raw = adata.raw.to_adata()[selection].copy()

# 3. Clean up the categories (important for BBKNN)
adata_endothelial.obs['patient'] = adata_endothelial.obs['patient'].cat.remove_unused_categories()

# 4. Now run your pipeline as normal
sc.pp.highly_variable_genes(adata_endothelial, n_top_genes=2000, batch_key='patient')
sc.tl.pca(adata_endothelial, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

import bbknn
bbknn.bbknn(
    adata_endothelial,
    batch_key="patient",
    use_rep="X_pca",
    neighbors_within_batch=5,
    n_pcs=30
)

sc.tl.umap(adata_endothelial, random_state=27)
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_endothelial, resolution=0.25, key_added="endothelial_subclusters", random_state=27)

sc.pl.umap(adata_endothelial, color=["condition", "patient", "endothelial_subclusters", "tumor_grade"])

In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_endothelial, 'endothelial_subclusters', method='wilcoxon')
sc.pl.rank_genes_groups(adata_endothelial, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_endothelial.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

In [ ]:
# Marker-set scores on the subset UMAP, used to spot contaminating cell types.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata_endothelial, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata_endothelial,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

### RNA velocity

In [ ]:
# Rename the kb-python layers for scVelo: nascent -> unspliced, mature + ambiguous -> spliced.
# (Requires .h5ad files made with `kb count --workflow nac`, as in the SLURM script.)
# Map Nascent to Unspliced
adata_endothelial.layers['unspliced'] = adata_endothelial.layers['nascent'] # Nascent reads definitively contain intronic sequences (unspliced)

# Map Mature (+ Ambiguous) to Spliced. Adding ambiguous to spliced to recover reads that overlap both an exon and a boundary.
adata_endothelial.layers['spliced'] = adata_endothelial.layers['mature'] + adata_endothelial.layers['ambiguous'] # Mature reads are purely exonic (spliced)

# Remove the old keys so scVelo doesn't get confused
del adata_endothelial.layers['nascent']
del adata_endothelial.layers['mature']
del adata_endothelial.layers['ambiguous']

# Verify the proportions
scv.utils.show_proportions(adata_endothelial)
scv.pl.proportions(adata_endothelial)

In [ ]:
# --- 2. Velocity-Specific Preprocessing ---
# This filters specifically for genes with sufficient spliced/unspliced counts
# >>> TUNE: minimum spliced and unspliced counts a gene needs to be kept
scv.pp.filter_genes(adata_endothelial, min_shared_counts=10)

# If your adata is already normalized/log-transformed, scVelo can 
# usually detect this. But we run this to ensure the layers are handled correctly.
scv.pp.normalize_per_cell(adata_endothelial)

In [ ]:
# Restore log-normalised expression and check its range (X is rebuilt in the next cell).
# restore log-normalised X from .raw
adata_endothelial.X = adata_endothelial.raw.to_adata()[:, adata_endothelial.var_names].X.copy()
print("X min/max:", adata_endothelial.X.min(), adata_endothelial.X.max())

In [ ]:
# Rebuild expression from the spliced + unspliced layers, log-transform and re-select variable genes for velocity.
adata_endothelial.X = (adata_endothelial.layers['spliced']
                         + adata_endothelial.layers['unspliced']).copy()

sc.pp.log1p(adata_endothelial)
print("X min/max:", adata_endothelial.X.min(), adata_endothelial.X.max())  # expect ~0-6

sc.pp.highly_variable_genes(adata_endothelial, flavor="seurat",
                            n_top_genes=3000, batch_key="patient")
adata_endothelial.raw = adata_endothelial
print("HVGs:", int(adata_endothelial.var["highly_variable"].sum()))

In [ ]:
# --- 3. The "BBKNN Preservation" Step ---
# DO NOT run sc.pp.neighbors or sc.pp.pca here. 
# We want to use the integrated graph created by BBKNN earlier.

print("Computing moments using the BBKNN-integrated graph...")
scv.pp.moments(adata_endothelial, n_pcs=None, n_neighbors=None)

In [ ]:
import tqdm
import ipywidgets
# --- 4. Compute Velocity ---
# Now calculate the actual velocity vectors
scv.tl.velocity(adata_endothelial, mode="stochastic") # 'stochastic' is faster/stable for first runs
scv.tl.velocity_graph(adata_endothelial, n_jobs=8)

In [ ]:
# Number of genes used for velocity (very few genes = unreliable velocity).
print("velocity_genes:", int(adata_endothelial.var['velocity_genes'].sum()))

In [ ]:
# Project velocities onto a UMAP embedding
scv.tl.velocity_embedding(adata_endothelial, basis='umap')

In [ ]:
# Velocity streamlines on the UMAP.
scv.pl.velocity_embedding_stream(
    adata_endothelial,
    basis="umap",
    color=["endothelial_subclusters", "patient"],
    legend_loc="right margin",
    title="RNA Velocity of ccRCC Atlas",
    smooth=0.9,
    size=450,
    alpha=0.6
)

In [ ]:
# Velocity confidence per cell, and the genes that drive velocity in each sub-cluster.
# Identify genes that are "speeding up" along the trajectory - genes that are actively being transcribed at the moment of transition
scv.tl.velocity_confidence(adata_endothelial)

scv.tl.rank_velocity_genes(adata_endothelial, groupby="endothelial_subclusters")

In [ ]:
# Velocity confidence and speed per cell on the UMAP.
scv.pl.scatter(adata_endothelial, 
               color=['velocity_confidence', 'velocity_length'], 
               cmap='coolwarm', 
               perc=[5, 95])

In [ ]:
# Table of the top 10 velocity genes per sub-cluster.
top_genes = pd.DataFrame(adata_endothelial.uns['rank_velocity_genes']['names'])
top_genes.head(10)

In [ ]:
# Phase portraits: unspliced vs spliced RNA for selected genes.
# >>> CHANGE FOR YOUR DATA: genes to show (pick from the velocity-gene table above)
target_genes = ["ESM1", "ANGPT2", "VEGFC"]

# Use scv.pl.scatter to generate phase portraits
# 'color' allows you to see where each cluster sits on the curve
scv.pl.scatter(adata_endothelial, 
               basis=target_genes, 
               color=['endothelial_subclusters'],
               frameon=False,
               ncols=2)

In [ ]:
# Project the velocity arrows for these specific genes onto the UMAP
scv.pl.velocity(adata_endothelial, 
                var_names=target_genes, 
                color='endothelial_subclusters')

### Trajectory and cell fate

In [ ]:
# Velocity pseudotime: each cell's position along the inferred trajectory (0 = start, 1 = end).
scv.tl.velocity_pseudotime(adata_endothelial)
scv.pl.scatter(adata_endothelial, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
# PAGA: simplified map of how strongly the sub-clusters are connected.
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

# 1. Compute PAGA connectivity between sub-clusters
sc.tl.paga(adata_endothelial, groups='endothelial_subclusters')

# 2. Plot it (this shows the 'skeleton' of your data)
scv.pl.paga(adata_endothelial, basis='umap', size=50, alpha=.1, 
            min_edge_width=2, node_size_scale=1.5)

In [ ]:
# Set root in cluster 0 using DNASE1L3 (a marker of the starting endothelial cluster)
# >>> CHANGE FOR YOUR DATA: starting cluster ("0") and a marker gene of it ("DNASE1L3");
#     the root is the cell in that cluster with the highest expression of the gene
cluster0_mask = adata_endothelial.obs["endothelial_subclusters"] == "0"
DNASE1L3_expr = adata_endothelial[cluster0_mask, "DNASE1L3"].X
if hasattr(DNASE1L3_expr, "toarray"):
    DNASE1L3_expr = DNASE1L3_expr.toarray().flatten()
adata_endothelial.uns["iroot"] = np.where(cluster0_mask)[0][np.argmax(DNASE1L3_expr)]

# Rerun diffmap and DPT
sc.tl.diffmap(adata_endothelial, n_comps=15)
sc.tl.dpt(adata_endothelial, n_dcs=10)

# Visualize
sc.pl.umap(adata_endothelial, color=["dpt_pseudotime"], cmap="gnuplot")

In [ ]:
# CellRank transition model (80% RNA velocity + 20% transcriptional similarity) and its spectrum.
import cellrank as cr
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel, VelocityKernel
from cellrank.estimators import GPCCA

# Initialize the kernels
vk = VelocityKernel(adata_endothelial).compute_transition_matrix()
ck = ConnectivityKernel(adata_endothelial).compute_transition_matrix()

# Combine them (80% velocity, 20% transcriptomic similarity)
# >>> TUNE: weighting between velocity and similarity
combined_kernel = 0.8 * vk + 0.2 * ck

# Rerun estimator
g = GPCCA(combined_kernel)
# >>> TUNE: number of components to compute
g.compute_schur(n_components=12)
g.plot_spectrum(real_only=True)

In [ ]:
# Compute macrostates - coarse grained states of the trajectory
# >>> TUNE: number of macrostates; a gap in the spectrum plot above suggests a good value
g.compute_macrostates(n_states=12, cluster_key="endothelial_subclusters")
g.plot_macrostates(which="all", discrete=False)

# Check how many cells are in each macrostate
print(g.macrostates.value_counts())

# Check macrostate memberships - should show soft assignments for all cells
print(g.macrostates_memberships)

# Plot memberships instead of hard assignments
g.plot_macrostates(which="all", discrete=False)  # continuous membership

In [ ]:
# --- Unsupervised check: let CellRank find terminal states from the graph itself ---
g_unsup = cr.estimators.GPCCA(combined_kernel)   # same kernel you already built

# 1. Compute macrostates. Ask for a few more than you expect (start + end states + a couple).
g_unsup.compute_schur(n_components=20)
# >>> TUNE: number of macrostates
g_unsup.compute_macrostates(n_states=6, cluster_key="endothelial_subclusters")

# 2. Let CellRank DECIDE which macrostates are terminal (no manual input)
g_unsup.predict_terminal_states()          # data-driven, based on coarse-grained stationarity
g_unsup.predict_initial_states(allow_overlap=True)           # likewise for the root

# 3. Inspect what it chose
print("Predicted terminal states:")
print(g_unsup.terminal_states)             # which cells/clusters it flagged as sinks
print("Predicted initial states:")
print(g_unsup.initial_states)

g_unsup.plot_macrostates(which="terminal", basis="umap", legend_loc="right")
g_unsup.plot_macrostates(which="initial",  basis="umap", legend_loc="right")

In [ ]:
# Set start and end states, then compute each cell's probability of reaching each end state.
# 7. Set initial (start) and terminal (end) states
#g.predict_terminal_states()
# >>> CHANGE FOR YOUR DATA: macrostate names from the plots above (initial = start, terminal = end states)
g.set_initial_states(states=["0"])
g.set_terminal_states(states=["1_1", "1_4"])

g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot=False)
# Then verify what was identified
#sc.pl.umap(adata, color=["macrostates_fwd", "terminal_states"])
#print("Terminal states identified:", adata.obs["terminal_states"].cat.categories.tolist())
#g.plot_macrostates(which="terminal", legend_loc="right")

In [ ]:
# Dynamical velocity model -> latent time, a second estimate of each cell's position along the trajectory.
scv.pp.moments(adata_endothelial)

# 2. Run the dynamical model
# This fits the transcriptional dynamics for each gene
scv.tl.recover_dynamics(adata_endothelial)

# 3. Compute velocity using the dynamical model
# 'mode' must be explicitly set to 'dynamical'
scv.tl.velocity(adata_endothelial, mode='dynamical')

# 4. Now you can compute latent time
scv.tl.latent_time(adata_endothelial)

# 5. Finally, plot it
import scanpy as sc
sc.pl.umap(adata_endothelial, color='latent_time', cmap='viridis')
scv.pl.velocity_embedding_stream(adata_endothelial, basis='umap', color='latent_time', title='Kinetic Age and Trajectory')

### Angiogenic signature

In [ ]:
# Score cells for an angiogenesis gene set and plot it on the UMAP.
import scanpy as sc

# >>> CHANGE FOR YOUR DATA: gene set of interest
angiogenic_genes = [
    "ESM1", "APLN", "ANGPT2", "INSR",
    "CXCR4", "PGF", "NID2"
]

# Keep only genes present in the dataset
angiogenic_genes = [
    g for g in angiogenic_genes
    if g in adata_endothelial.var_names
]

# Calculate gene-set score
sc.tl.score_genes(
    adata_endothelial,
    gene_list=angiogenic_genes,
    score_name="Angiogenic"
)

# Plot on UMAP
sc.pl.umap(
    adata_endothelial,
    color="Angiogenic",
    cmap="viridis",
    size=15
)

### Lineage drivers

In [ ]:
# Lineage drivers: genes correlated with the probability of reaching each end state.
# 6. Find genes driving each lineage
lineage_drivers = g.compute_lineage_drivers(return_drivers=True)

# 7. View the top 20 drivers for this terminal state
# >>> CHANGE FOR YOUR DATA: "1_1" = terminal macrostate name
lineage_drivers.sort_values(by="1_1_corr", ascending=False).head(20)

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "1_1" = terminal macrostate name
g.plot_lineage_drivers(lineage="1_1", n_genes=20)

In [ ]:
# Volcano-style plot: correlation with fate probability vs significance; top 20 drivers labelled.
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# >>> CHANGE FOR YOUR DATA: "1_1" = terminal macrostate name
lin = "1_1"
df = adata_endothelial.varm["terminal_lineage_drivers"].copy()
df = df[[f"{lin}_corr", f"{lin}_qval"]].dropna()
df.columns = ["corr", "qval"]
df["neglog_q"] = -np.log10(df["qval"].clip(lower=1e-300))

top20 = df[df["corr"] > 0].nlargest(20, "neglog_q")

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(df["corr"], df["neglog_q"], s=6, c="lightgrey", alpha=0.5, rasterized=True)
ax.scatter(top20["corr"], top20["neglog_q"], s=30, c="crimson", zorder=3)

ax.set_yscale("log")
# >>> TUNE: axis limits for this lineage (cosmetic)
ax.set_ylim(1, 450)             # crop the empty upper region
ax.set_xlim(0, 0.6)        # crop the empty left tail (negative correlations)

texts = [ax.text(r["corr"], r["neglog_q"], g, fontsize=18, zorder=4)
         for g, r in top20.iterrows()]
adjust_text(texts, ax=ax,
            expand_text=(1.6, 1.8),
            force_text=(0.6, 1.2),
            arrowprops=dict(arrowstyle="-", lw=0.6, color="grey"))

ax.axvline(0, ls="--", lw=0.8, c="black")
ax.set_xlabel(f"Correlation with lineage {lin} fate probability", fontsize=18)
ax.set_ylabel("-log10(q-value)  [log scale]", fontsize=18)
ax.set_title(f"Lineage {lin} drivers", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 drivers for another end state.
# >>> CHANGE FOR YOUR DATA: "1_4" = terminal macrostate name
lineage_drivers.sort_values(by="1_4_corr", ascending=False).head(20)

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "1_4" = terminal macrostate name
g.plot_lineage_drivers(lineage="1_4", n_genes=20)

In [ ]:
# Volcano-style plot: correlation with fate probability vs significance; top 20 drivers labelled.
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# >>> CHANGE FOR YOUR DATA: "1_4" = terminal macrostate name
lin = "1_4"
df = adata_endothelial.varm["terminal_lineage_drivers"].copy()
df = df[[f"{lin}_corr", f"{lin}_qval"]].dropna()
df.columns = ["corr", "qval"]
df["neglog_q"] = -np.log10(df["qval"].clip(lower=1e-300))

top20 = df[df["corr"] > 0].nlargest(20, "neglog_q")

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(df["corr"], df["neglog_q"], s=6, c="lightgrey", alpha=0.5, rasterized=True)
ax.scatter(top20["corr"], top20["neglog_q"], s=30, c="crimson", zorder=3)

ax.set_yscale("log")
# >>> TUNE: axis limits for this lineage (cosmetic)
ax.set_ylim(1, 450)             # crop the empty upper region
ax.set_xlim(0, 0.55)        # crop the empty left tail (negative correlations)

texts = [ax.text(r["corr"], r["neglog_q"], g, fontsize=18, zorder=4)
         for g, r in top20.iterrows()]
adjust_text(texts, ax=ax,
            expand_text=(1.6, 1.8),
            force_text=(0.6, 1.2),
            arrowprops=dict(arrowstyle="-", lw=0.6, color="grey"))

ax.axvline(0, ls="--", lw=0.8, c="black")
ax.set_xlabel(f"Correlation with lineage {lin} fate probability", fontsize=18)
ax.set_ylabel("-log10(q-value)  [log scale]", fontsize=18)
ax.set_title(f"Lineage {lin} drivers", fontsize=14)
plt.tight_layout()
plt.show()

## B3. Fibroblast / CAF compartment
Same subsetting and velocity steps as B1; this section ends with diffusion pseudotime (no CellRank fate analysis).
### Subset and re-cluster

In [ ]:
# 1. Perform the subsetting selection
# >>> CHANGE FOR YOUR DATA: cell-type label(s) (names from `markers`) that make up this compartment
selection = adata.obs["cell_type"].isin([
    "Fibroblasts", "Myofibroblasts (CAFs)", "Pericytes/Smooth Muscle"
    ]).values

# 2. MANUALLY REBUILD the object to drop broken obsp/uns
# This takes the expression matrix (X), observations (obs), and genes (var) 
# and puts them into a brand new, clean container.
adata_fibro = sc.AnnData(
    X=adata[selection].X.copy(),
    obs=adata[selection].obs.copy(),
    var=adata.var.copy(),
    layers={key: layer[selection].copy() for key, layer in adata.layers.items()}
)

adata_fibro.raw = adata.raw.to_adata()[selection].copy()

# 3. Clean up the categories (important for BBKNN)
adata_fibro.obs['patient'] = adata_fibro.obs['patient'].cat.remove_unused_categories()

# 4. Now run your pipeline as normal
sc.pp.highly_variable_genes(adata_fibro, n_top_genes=2000, batch_key='patient')
sc.tl.pca(adata_fibro, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

import bbknn
bbknn.bbknn(
    adata_fibro,
    batch_key="patient",
    use_rep="X_pca",
    neighbors_within_batch=5,
    n_pcs=30
)

sc.tl.umap(adata_fibro, random_state=27)
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_fibro, resolution=0.35, key_added="fibro_subclusters", random_state=27)

sc.pl.umap(adata_fibro, color=["condition", "patient", "fibro_subclusters"])

In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_fibro, 'fibro_subclusters', method='wilcoxon')
sc.pl.rank_genes_groups(adata_fibro, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_fibro.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

In [ ]:
# Marker-set scores on the subset UMAP (scores carried over from the full dataset).
sc.pl.umap(
    adata_fibro,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

In [ ]:
# Keep the selected sub-cluster(s) and re-embed.
# >>> CHANGE FOR YOUR DATA: sub-cluster numbers to keep; choose them from the plots above
adata_fibro_clean = adata_fibro[adata_fibro.obs['fibro_subclusters'].isin(["6"])].copy()

# 1. HVGs: find the genes that vary within this compartment
sc.pp.highly_variable_genes(adata_fibro_clean, n_top_genes=2000, batch_key='patient')

# 2. PCA: Define the variation space for just this lineage
sc.tl.pca(adata_fibro_clean, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

# 3. BBKNN: Final integration to ensure patient-specific effects are gone
bbknn.bbknn(
    adata_fibro_clean,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

# 4. UMAP: This should now show a beautiful, continuous trajectory
sc.tl.umap(adata_fibro_clean, random_state=27)
sc.pl.umap(adata_fibro_clean, color=["condition", "patient", "fibro_subclusters", "tumor_grade"])

In [ ]:
# Run Leiden clustering on the refined BBKNN graph
# key_added prevents overwriting your previous 'fibro_subclusters'
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_fibro_clean, resolution=0.9, key_added='fibro_fine_grain')

# Visualize the new clusters
sc.pl.umap(adata_fibro_clean, color=['fibro_fine_grain', 'patient', 'tumor_grade'])

In [ ]:
# Second clean-up round, then re-embed.
# >>> CHANGE FOR YOUR DATA: sub-cluster numbers to keep; choose them from the plots above
adata_fibro_clean_clean = adata_fibro_clean[adata_fibro_clean.obs['fibro_fine_grain'].isin(["1", "2", "3", "5", "6"])].copy()

# 1. HVGs: find the genes that vary within this compartment
sc.pp.highly_variable_genes(adata_fibro_clean_clean, n_top_genes=2000, batch_key='patient')

# 2. PCA: Define the variation space for just this lineage
sc.tl.pca(adata_fibro_clean_clean, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

# 3. BBKNN: Final integration to ensure patient-specific effects are gone
bbknn.bbknn(
    adata_fibro_clean_clean,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=2,   #Number of neighbors to use within each batch
    n_pcs=30
)

# 4. UMAP: This should now show a beautiful, continuous trajectory
sc.tl.umap(adata_fibro_clean_clean, random_state=27)
sc.pl.umap(adata_fibro_clean_clean, color=["condition", "patient", "fibro_fine_grain", "tumor_grade"])

In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_fibro_clean_clean, 'fibro_fine_grain', method='wilcoxon')
sc.pl.rank_genes_groups(adata_fibro_clean_clean, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_fibro_clean_clean.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

In [ ]:
# Marker-set scores on the subset UMAP, used to spot contaminating cell types.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata_fibro_clean_clean, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata_fibro_clean_clean,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

### RNA velocity

In [ ]:
# Rename the kb-python layers for scVelo: nascent -> unspliced, mature + ambiguous -> spliced.
# (Requires .h5ad files made with `kb count --workflow nac`, as in the SLURM script.)
# Map Nascent to Unspliced
adata_fibro_clean_clean.layers['unspliced'] = adata_fibro_clean_clean.layers['nascent'] # Nascent reads definitively contain intronic sequences (unspliced)

# Map Mature (+ Ambiguous) to Spliced. Adding ambiguous to spliced to recover reads that overlap both an exon and a boundary.
adata_fibro_clean_clean.layers['spliced'] = adata_fibro_clean_clean.layers['mature'] + adata_fibro_clean_clean.layers['ambiguous'] # Mature reads are purely exonic (spliced)

# Remove the old keys so scVelo doesn't get confused
del adata_fibro_clean_clean.layers['nascent']
del adata_fibro_clean_clean.layers['mature']
del adata_fibro_clean_clean.layers['ambiguous']

# Verify the proportions
scv.utils.show_proportions(adata_fibro_clean_clean)
scv.pl.proportions(adata_fibro_clean_clean)

In [ ]:
# --- 2. Velocity-Specific Preprocessing ---
# This filters specifically for genes with sufficient spliced/unspliced counts
# >>> TUNE: minimum spliced and unspliced counts a gene needs to be kept
scv.pp.filter_genes(adata_fibro_clean_clean, min_shared_counts=10)

# If your adata is already normalized/log-transformed, scVelo can 
# usually detect this. But we run this to ensure the layers are handled correctly.
scv.pp.normalize_per_cell(adata_fibro_clean_clean)


In [ ]:
# Restore log-normalised expression and check its range (X is rebuilt in the next cell).
# restore log-normalised X from .raw
adata_fibro_clean_clean.X = adata_fibro_clean_clean.raw.to_adata()[:, adata_fibro_clean_clean.var_names].X.copy() 
print("X min/max:", adata_fibro_clean_clean.X.min(), adata_fibro_clean_clean.X.max())

In [ ]:
# Rebuild expression from the spliced + unspliced layers, log-transform and re-select variable genes for velocity.
adata_fibro_clean_clean.X = (adata_fibro_clean_clean.layers['spliced']
                         + adata_fibro_clean_clean.layers['unspliced']).copy()

sc.pp.log1p(adata_fibro_clean_clean)
print("X min/max:", adata_fibro_clean_clean.X.min(), adata_fibro_clean_clean.X.max())  # expect ~0-6

sc.pp.highly_variable_genes(adata_fibro_clean_clean, flavor="seurat",
                            n_top_genes=3000, batch_key="patient")
adata_fibro_clean_clean.raw = adata_fibro_clean_clean
print("HVGs:", int(adata_fibro_clean_clean.var["highly_variable"].sum()))

In [ ]:
# --- 3. The "BBKNN Preservation" Step ---
# DO NOT run sc.pp.neighbors or sc.pp.pca here. 
# We want to use the integrated graph created by BBKNN earlier.

print("Computing moments using the BBKNN-integrated graph...")
scv.pp.moments(adata_fibro_clean_clean, n_pcs=None, n_neighbors=None)


In [ ]:
import tqdm
import ipywidgets
# --- 4. Compute Velocity ---
# Now calculate the actual velocity vectors
scv.tl.velocity(adata_fibro_clean_clean, mode="stochastic") # 'stochastic' is faster/stable for first runs
scv.tl.velocity_graph(adata_fibro_clean_clean, n_jobs=8)


In [ ]:
# Number of genes used for velocity (very few genes = unreliable velocity).
print("velocity_genes:", int(adata_fibro_clean_clean.var['velocity_genes'].sum()))

In [ ]:
# Project velocities onto a UMAP embedding
scv.tl.velocity_embedding(adata_fibro_clean_clean, basis='umap')

In [ ]:
# Velocity streamlines on the UMAP.
scv.pl.velocity_embedding_stream(
    adata_fibro_clean_clean,
    basis="umap",
    color="fibro_fine_grain",
    legend_loc="right margin",
    title="RNA Velocity of ccRCC Atlas",
    smooth=0.9,
    size=450,
    alpha=0.6
)

In [ ]:
# Velocity confidence per cell, and the genes that drive velocity in each sub-cluster.
# Identify genes that are "speeding up" along the trajectory - genes that are actively being transcribed at the moment of transition
scv.tl.velocity_confidence(adata_fibro_clean_clean)
scv.tl.rank_velocity_genes(adata_fibro_clean_clean, groupby='fibro_fine_grain')

In [ ]:
# Velocity confidence and speed per cell on the UMAP.
scv.pl.scatter(adata_fibro_clean_clean, 
               color=['velocity_confidence', 'velocity_length'], 
               cmap='coolwarm', 
               perc=[5, 95])

In [ ]:
# Table of the top 10 velocity genes per sub-cluster.
# Extract the ranked names into a DataFrame
top_genes = pd.DataFrame(adata_fibro_clean_clean.uns['rank_velocity_genes']['names'])
top_genes.head(10)

### Diffusion pseudotime

In [ ]:
# Score myCAF and resting-fibroblast gene sets to orient the trajectory.
# >>> CHANGE FOR YOUR DATA: gene sets of interest
mycaf = ["FAP","POSTN","COL1A1","COL1A2","COL3A1","FN1","BGN","VCAN","ASPN"]
resting = ["CCDC80","DCN","C3","ABI3BP"]

sc.tl.score_genes(adata_fibro_clean_clean, mycaf, score_name="myCAF_score")
sc.tl.score_genes(adata_fibro_clean_clean, resting, score_name="resting_score")

sc.pl.umap(adata_fibro_clean_clean, color=["myCAF_score","resting_score","fibro_fine_grain"])
sc.pl.violin(adata_fibro_clean_clean, ["myCAF_score","resting_score"], groupby="fibro_fine_grain")

In [ ]:
# Set root in cluster 5 using CCDC80 (a resting-fibroblast marker, see the gene sets above)
# >>> CHANGE FOR YOUR DATA: starting cluster ("5") and a marker gene of it ("CCDC80");
#     the root is the cell in that cluster with the highest expression of the gene
cluster5_mask = adata_fibro_clean_clean.obs["fibro_fine_grain"] == "5"
CFD_expr = adata_fibro_clean_clean[cluster5_mask, "CCDC80"].X
if hasattr(CFD_expr, "toarray"):
    CFD_expr = CFD_expr.toarray().flatten()
adata_fibro_clean_clean.uns["iroot"] = np.where(cluster5_mask)[0][np.argmax(CFD_expr)]

# Rerun diffmap and DPT
sc.tl.diffmap(adata_fibro_clean_clean, n_comps=15)
sc.tl.dpt(adata_fibro_clean_clean, n_dcs=10)

# Visualize
sc.pl.umap(adata_fibro_clean_clean, color=["dpt_pseudotime"], cmap="gnuplot")

## B4. Myeloid compartment: monocyte → macrophage (TAM)
Same steps as B1.
### Subset and re-cluster

In [ ]:
# 1. Perform the subsetting selection
# >>> CHANGE FOR YOUR DATA: cell-type label(s) (names from `markers`) that make up this compartment
selection = adata.obs["cell_type"].isin([
    "Monocytes", "Macrophages (General)", "Macrophages (M2-like/TAM)"
]).values

# 2. MANUALLY REBUILD the object to drop broken obsp/uns
# This takes the expression matrix (X), observations (obs), and genes (var) 
# and puts them into a brand new, clean container.
adata_myeloid = sc.AnnData(
    X=adata[selection].X.copy(),
    obs=adata[selection].obs.copy(),
    var=adata.var.copy(),
    layers={key: layer[selection].copy() for key, layer in adata.layers.items()}
)

adata_myeloid.raw = adata.raw.to_adata()[selection].copy()

# 3. Clean up the categories (important for BBKNN)
adata_myeloid.obs['patient'] = adata_myeloid.obs['patient'].cat.remove_unused_categories()

# 4. Now run your pipeline as normal
sc.pp.highly_variable_genes(adata_myeloid, n_top_genes=2000, batch_key='patient')
sc.tl.pca(adata_myeloid, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

import bbknn
bbknn.bbknn(
    adata_myeloid,
    batch_key="patient",
    use_rep="X_pca",
    neighbors_within_batch=5,
    n_pcs=30
)

sc.tl.umap(adata_myeloid, random_state=27)
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_myeloid, resolution=0.2, key_added="myeloid_subclusters", random_state=27)

sc.pl.umap(adata_myeloid, color=["condition", "patient", "myeloid_subclusters"])

In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_myeloid, 'myeloid_subclusters', method='wilcoxon')
sc.pl.rank_genes_groups(adata_myeloid, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_myeloid.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

In [ ]:
# Marker-set scores on the subset UMAP, used to spot contaminating cell types.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata_myeloid, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata_myeloid,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

In [ ]:
# Remove contaminating sub-clusters and re-embed.
# >>> CHANGE FOR YOUR DATA: sub-cluster numbers to keep; choose them from the plots above
adata_myeloid_clean = adata_myeloid[adata_myeloid.obs['myeloid_subclusters'].isin(["1", "2", "5", "6", "7", "9", "10"])].copy()

# 1. HVGs: find the genes that vary within this compartment
sc.pp.highly_variable_genes(adata_myeloid_clean, n_top_genes=2000, batch_key='patient')

# 2. PCA: Define the variation space for just this lineage
sc.tl.pca(adata_myeloid_clean, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

# 3. BBKNN: Final integration to ensure patient-specific effects are gone
bbknn.bbknn(
    adata_myeloid_clean,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

# 4. UMAP: This should now show a beautiful, continuous trajectory
sc.tl.umap(adata_myeloid_clean, random_state=27)
sc.pl.umap(adata_myeloid_clean, color=["condition", "patient", "myeloid_subclusters", "tumor_grade"])

In [ ]:
# Re-run the UMAP (repeat of the previous cell).
sc.tl.umap(adata_myeloid_clean, random_state=27)
sc.pl.umap(adata_myeloid_clean, color=["condition", "patient", "myeloid_subclusters", "tumor_grade"])

In [ ]:
# >>> TUNE: clustering resolution
sc.tl.leiden(adata_myeloid_clean, resolution=0.2, key_added='myeloid_fine_grain')

# Visualize the new clusters
sc.pl.umap(adata_myeloid_clean, color=['myeloid_fine_grain', 'patient', 'tumor_grade'])

In [ ]:
# Second clean-up round, then re-embed.
# >>> CHANGE FOR YOUR DATA: sub-cluster numbers to keep; choose them from the plots above
adata_myeloid_clean = adata_myeloid_clean[adata_myeloid_clean.obs['myeloid_fine_grain'].isin(["0", "1", "2", "3", "4", "5", "6", "7", "8", "10", "11", "12"])].copy()

# 1. HVGs: find the genes that vary within this compartment
sc.pp.highly_variable_genes(adata_myeloid_clean, n_top_genes=2000, batch_key='patient')

# 2. PCA: Define the variation space for just this lineage
sc.tl.pca(adata_myeloid_clean, n_comps=30, use_highly_variable=True, svd_solver="arpack", random_state=27)

# 3. BBKNN: Final integration to ensure patient-specific effects are gone
bbknn.bbknn(
    adata_myeloid_clean,
    batch_key="patient",    #Correcting for patient-specific effects
    use_rep="X_pca",
    neighbors_within_batch=5,   #Number of neighbors to use within each batch
    n_pcs=30
)

# 4. UMAP: This should now show a beautiful, continuous trajectory
sc.tl.umap(adata_myeloid_clean, random_state=27)
sc.pl.umap(adata_myeloid_clean, color=["condition", "patient", "myeloid_fine_grain", "tumor_grade"])

In [ ]:
# Marker-set scores on the subset UMAP, used to spot contaminating cell types.
for celltype, genes in markers.items():
    sc.tl.score_genes(adata_myeloid_clean, gene_list=genes, score_name=celltype)

sc.pl.umap(
    adata_myeloid_clean,
    color=list(markers.keys()),
    ncols=3,
    cmap="viridis"
)

In [ ]:
# Selected genes on the UMAP.
# >>> CHANGE FOR YOUR DATA: genes to plot
sc.pl.umap(
    adata_myeloid_clean,
    color=[
        "SPI1",        # PU1
        "CD163",   
    ],
    cmap="viridis",
    ncols=4
)

In [ ]:
# Marker genes for each sub-cluster (Wilcoxon test).
sc.tl.rank_genes_groups(adata_myeloid_clean, 'myeloid_fine_grain', method='wilcoxon')
sc.pl.rank_genes_groups(adata_myeloid_clean, n_genes=20, sharey=False)

In [ ]:
# Table of the top 20 marker genes per sub-cluster (one column per cluster).
# Extract results into a clean DataFrame
result = adata_myeloid_clean.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame({
    group: result['names'][group]
    for group in groups
})

df.head(20)

### RNA velocity

In [ ]:
# Rename the kb-python layers for scVelo: nascent -> unspliced, mature + ambiguous -> spliced.
# (Requires .h5ad files made with `kb count --workflow nac`, as in the SLURM script.)
# Map Nascent to Unspliced
adata_myeloid_clean.layers['unspliced'] = adata_myeloid_clean.layers['nascent'] # Nascent reads definitively contain intronic sequences (unspliced)

# Map Mature (+ Ambiguous) to Spliced. Adding ambiguous to spliced to recover reads that overlap both an exon and a boundary.
adata_myeloid_clean.layers['spliced'] = adata_myeloid_clean.layers['mature'] + adata_myeloid_clean.layers['ambiguous'] # Mature reads are purely exonic (spliced)

# Remove the old keys so scVelo doesn't get confused
del adata_myeloid_clean.layers['nascent']
del adata_myeloid_clean.layers['mature']
del adata_myeloid_clean.layers['ambiguous']

# Verify the proportions
scv.utils.show_proportions(adata_myeloid_clean)
scv.pl.proportions(adata_myeloid_clean)

In [ ]:
# --- 2. Velocity-Specific Preprocessing ---
# This filters specifically for genes with sufficient spliced/unspliced counts
# >>> TUNE: minimum spliced and unspliced counts a gene needs to be kept
scv.pp.filter_genes(adata_myeloid_clean, min_shared_counts=10)

# If your adata is already normalized/log-transformed, scVelo can 
# usually detect this. But we run this to ensure the layers are handled correctly.
scv.pp.normalize_per_cell(adata_myeloid_clean)

In [144]:
# Inactive (commented out): earlier fix, kept for reference.
# Subset inherited SCALED .X from the scaling step in Part A, which starved HVG selection (-> only 10 velocity genes).
# Restore log-normalized expression from .raw and re-flag HVGs so velocity gets a full candidate pool.
#adata_myeloid_clean.X = adata_myeloid_clean.raw.to_adata()[:, adata_myeloid_clean.var_names].X.copy()
#sc.pp.highly_variable_genes(adata_myeloid_clean, flavor="seurat", n_top_genes=2000, batch_key="patient")
#print("HVGs after fix:", int(adata_myeloid_clean.var["highly_variable"].sum()))

In [ ]:
# Restore log-normalised expression and check its range (X is rebuilt in the next cell).
# restore log-normalised X from .raw
adata_myeloid_clean.X = adata_myeloid_clean.raw.to_adata()[:, adata_myeloid_clean.var_names].X.copy()
print("X min/max:", adata_myeloid_clean.X.min(), adata_myeloid_clean.X.max())

In [ ]:
# Rebuild expression from the spliced + unspliced layers, log-transform and re-select variable genes for velocity.
adata_myeloid_clean.X = (adata_myeloid_clean.layers['spliced']
                         + adata_myeloid_clean.layers['unspliced']).copy()

sc.pp.log1p(adata_myeloid_clean)
print("X min/max:", adata_myeloid_clean.X.min(), adata_myeloid_clean.X.max())  # expect ~0-6

sc.pp.highly_variable_genes(adata_myeloid_clean, flavor="seurat",
                            n_top_genes=3000, batch_key="patient")
adata_myeloid_clean.raw = adata_myeloid_clean
print("HVGs:", int(adata_myeloid_clean.var["highly_variable"].sum()))

In [ ]:
# --- 3. The "BBKNN Preservation" Step ---
# DO NOT run sc.pp.neighbors or sc.pp.pca here. 
# We want to use the integrated graph created by BBKNN earlier.

print("Computing moments using the BBKNN-integrated graph...")
scv.pp.moments(adata_myeloid_clean, n_pcs=None, n_neighbors=None)

In [ ]:
import tqdm
import ipywidgets
# --- 4. Compute Velocity ---
# Now calculate the actual velocity vectors
scv.tl.velocity(adata_myeloid_clean, mode="stochastic") # 'stochastic' is faster/stable for first runs
scv.tl.velocity_graph(adata_myeloid_clean, n_jobs=8)

In [ ]:
# Number of genes used for velocity (very few genes = unreliable velocity).
print("velocity_genes:", int(adata_myeloid_clean.var['velocity_genes'].sum()))

In [ ]:
# Which known marker genes made it into the velocity gene set?
vg = adata_myeloid_clean.var_names[adata_myeloid_clean.var['velocity_genes']]
print(list(vg))
# >>> CHANGE FOR YOUR DATA: marker genes to look for
print([g for g in ['TREM2','APOE','APOC1','LGALS3','C1QA','C1QB','CD68',
                   'LYVE1','FOLR2','MRC1','CD163','SIGLEC1','SERPINB2',
                   'S100A8','S100A9','VCAN','FCN1'] if g in vg])

In [ ]:
# Project velocities onto a UMAP embedding
scv.tl.velocity_embedding(adata_myeloid_clean, basis='umap')

In [ ]:
# Velocity streamlines on the UMAP, coloured by patient.
scv.pl.velocity_embedding_stream(
    adata_myeloid_clean,
    basis="umap",
    color="patient",
    legend_loc="right margin",
    title="RNA Velocity of ccRCC Atlas",
    smooth=0.9,
    size=450,
    alpha=0.6
)

In [ ]:
# Velocity confidence per cell, and the genes that drive velocity in each sub-cluster.
# Identify genes that are "speeding up" along the trajectory - genes that are actively being transcribed at the moment of transition
scv.tl.velocity_confidence(adata_myeloid_clean)

scv.tl.rank_velocity_genes(adata_myeloid_clean, groupby="myeloid_fine_grain")

In [ ]:
# Velocity confidence and speed per cell on the UMAP.
scv.pl.scatter(adata_myeloid_clean, 
               color=['velocity_confidence', 'velocity_length'], 
               cmap='coolwarm', 
               perc=[5, 95])

In [ ]:
# Table of the top 10 velocity genes per sub-cluster.
# Extract the ranked names into a DataFrame
top_genes = pd.DataFrame(adata_myeloid_clean.uns["rank_velocity_genes"]["names"])
top_genes.head(10)

In [ ]:
# Phase portraits: unspliced vs spliced RNA for selected genes.
# >>> CHANGE FOR YOUR DATA: genes to show (pick from the velocity-gene table above)
target_genes = ["SERPINB2", "ZEB2", "CD163", "LYVE1"]

# Use scv.pl.scatter to generate phase portraits
# 'color' allows you to see where each cluster sits on the curve
scv.pl.scatter(adata_myeloid_clean, 
               basis=target_genes, 
               color=['myeloid_fine_grain'],
               frameon=False,
               ncols=2)

In [ ]:
# Project the velocity arrows for these specific genes onto the UMAP
scv.pl.velocity(adata_myeloid_clean, 
                var_names=target_genes, 
                color='myeloid_fine_grain')

In [ ]:
# Velocity pseudotime: each cell's position along the inferred trajectory (0 = start, 1 = end).
scv.tl.velocity_pseudotime(adata_myeloid_clean)
scv.pl.scatter(adata_myeloid_clean, color='velocity_pseudotime', cmap='gnuplot')

### Cluster labels and TAM abundance

In [ ]:
# Dot plot of TREM2+ TAM marker genes per cluster.
# >>> CHANGE FOR YOUR DATA: genes to plot
sc.pl.dotplot(adata_myeloid_clean, ['TREM2','APOE','APOC1','LGALS3','GPNMB',
                                    'SPP1','CD9','FABP5'],
              groupby='myeloid_fine_grain', use_raw=True)

In [160]:
# Readable names for the myeloid clusters.
# >>> CHANGE FOR YOUR DATA: cluster number -> name
adata_myeloid_clean.obs['myeloid_label'] = adata_myeloid_clean.obs['myeloid_fine_grain'].map({
    '4': '4 TREM2⁺ LAM', '10': '10 Resident', '7': '7 Classical mono',
    '11': '11 Non-classical mono', '2': '2 C1Q⁺ APC mac', '3': '3 C1Q⁺ APC mac',
    '5': '5 SIGLEC1⁺ IFN mac', '6': '6 ISG⁺', '8': '8 Intermediate mono',
    '0': '0 Inflammatory mono', '1': '1 Macrophage', '12': '12 MHC-II⁺ mono'})

In [ ]:
# Figure: (a) marker evidence per cluster, (b) share of TREM2+ TAMs (cluster 4) among myeloid cells per group.
import matplotlib.pyplot as plt

obs = adata_myeloid_clean.obs

# --- define the sample grouping (patient + tissue) ---
print([c for c in obs.columns if obs[c].dtype.name in ('category', 'object')])
# >>> CHANGE FOR YOUR DATA: 'patient' pools tumour and normal; use 'sample' to split them
#     (the Normal/Tumour bar colours below rely on sample names ending in _N)
SAMPLE_COL = 'patient'   # <- set from the printout above

obs['group'] = obs[SAMPLE_COL].astype(str)
adata_myeloid_clean.obs['group'] = obs['group']
print(sorted(obs['group'].unique()))

# --- label the clusters ---
# >>> CHANGE FOR YOUR DATA: cluster number -> name
LABELS = {
    '0':  '0 Inflammatory mono',
    '1':  '1 Macrophage',
    '2':  '2 C1Q+ APC mac',
    '3':  '3 C1Q+ APC mac',
    '4':  '4 TREM2+ LAM',
    '5':  '5 SIGLEC1+ IFN mac',
    '6':  '6 ISG+ myeloid',
    '7':  '7 Classical mono',
    '8':  '8 Intermediate mono',
    '10': '10 Resident mac',
    '11': '11 Non-classical mono',
    '12': '12 MHC-II+ mono',
}
adata_myeloid_clean.obs['myeloid_label'] = (
    adata_myeloid_clean.obs['myeloid_fine_grain'].astype(str).map(LABELS))
adata_myeloid_clean.obs['myeloid_label'] = pd.Categorical(
    adata_myeloid_clean.obs['myeloid_label'],
    categories=[LABELS[k] for k in sorted(LABELS, key=int)])

# --- (a) marker evidence ---
# >>> CHANGE FOR YOUR DATA: marker genes per group
sc.pl.dotplot(adata_myeloid_clean,
              {'TREM2+ TAM': ['TREM2', 'APOE', 'APOC1', 'LGALS3', 'GPNMB'],
               'Resident':   ['LYVE1', 'FOLR2', 'MRC1', 'CD163', 'SELENOP'],
               'Monocyte':   ['S100A8', 'S100A9', 'VCAN', 'FCN1']},
              groupby='myeloid_fine_grain', use_raw=True,
              standard_scale='var', cmap='Reds', show=True)

# --- (b) cluster 4 share of myeloid, per sample ---
# >>> CHANGE FOR YOUR DATA: '4' = the TREM2+ TAM cluster number
comp = (obs.assign(is4=obs['myeloid_fine_grain'].eq('4'))
           .groupby('group', observed=True)['is4'].mean().mul(100)
           .reindex(sorted(obs['group'].unique())).to_frame('pct'))
comp['tissue'] = ['Normal' if g.endswith('_N') else 'Tumour' for g in comp.index]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(comp.index, comp['pct'],
       color=['#4C72B0' if t == 'Normal' else '#C44E52' for t in comp['tissue']])
ax.set_ylabel('TREM2$^+$ TAM (% of myeloid)')
ax.tick_params(axis='x', rotation=90)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ('#4C72B0', '#C44E52')]
ax.legend(handles, ['Normal', 'Tumour'], frameon=False, loc='upper left')
fig.tight_layout()
plt.show()

print(comp.round(2).to_string())

### Trajectory and cell fate

In [ ]:
# PAGA: simplified map of how strongly the sub-clusters are connected.
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

# 1. Compute PAGA connectivity between sub-clusters
sc.tl.paga(adata_myeloid_clean, groups='myeloid_subclusters')

# 2. Plot it (this shows the 'skeleton' of your data)
scv.pl.paga(adata_myeloid_clean, basis='umap', size=50, alpha=.1, 
            min_edge_width=2, node_size_scale=1.5)

In [ ]:
# CellRank transition model (80% RNA velocity + 20% transcriptional similarity) and its spectrum.
import cellrank as cr
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel, VelocityKernel
from cellrank.estimators import GPCCA

# Initialize the kernels
vk = VelocityKernel(adata_myeloid_clean).compute_transition_matrix()
ck = ConnectivityKernel(adata_myeloid_clean).compute_transition_matrix()

# Combine them (80% velocity, 20% transcriptomic similarity)
# >>> TUNE: weighting between velocity and similarity
combined_kernel = 0.8 * vk + 0.2 * ck

# Rerun estimator
g = GPCCA(combined_kernel)
# >>> TUNE: number of components to compute
g.compute_schur(n_components=15)
g.plot_spectrum(real_only=True)

In [ ]:
# Compute macrostates - coarse grained states of the trajectory
# >>> TUNE: number of macrostates; a gap in the spectrum plot above suggests a good value
g.compute_macrostates(n_states=15, cluster_key="myeloid_fine_grain")
g.plot_macrostates(which="all", discrete=False)

# Check how many cells are in each macrostate
print(g.macrostates.value_counts())

# Check macrostate memberships - should show soft assignments for all cells
print(g.macrostates_memberships)

# Plot memberships instead of hard assignments
g.plot_macrostates(which="all", discrete=False)  # continuous membership

In [ ]:
import cellrank as cr

# --- Unsupervised check: let CellRank find terminal states from the graph itself ---
g_unsup = cr.estimators.GPCCA(combined_kernel)   # same kernel you already built

# 1. Compute macrostates. Ask for a few more than you expect (start + end states + a couple).
g_unsup.compute_schur(n_components=20)
# >>> TUNE: number of macrostates
g_unsup.compute_macrostates(n_states=6, cluster_key="myeloid_fine_grain")

# 2. Let CellRank DECIDE which macrostates are terminal (no manual input)
g_unsup.predict_terminal_states()          # data-driven, based on coarse-grained stationarity
g_unsup.predict_initial_states(allow_overlap=True)           # likewise for the root

# 3. Inspect what it chose
print("Predicted terminal states:")
print(g_unsup.terminal_states)             # which cells/clusters it flagged as sinks
print("Predicted initial states:")
print(g_unsup.initial_states)

g_unsup.plot_macrostates(which="terminal", basis="umap", legend_loc="right")
g_unsup.plot_macrostates(which="initial",  basis="umap", legend_loc="right")

In [ ]:
# Set start and end states, then compute each cell's probability of reaching each end state.
# 7. Set initial (start) and terminal (end) states
#g.predict_terminal_states()
# >>> CHANGE FOR YOUR DATA: macrostate names from the plots above (initial = start, terminal = end states)
g.set_initial_states(states=["7"])
g.set_terminal_states(states=["4", "10", "3_2"])

g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot=False)
# Then verify what was identified
#sc.pl.umap(adata, color=["macrostates_fwd", "terminal_states"])
#print("Terminal states identified:", adata.obs["terminal_states"].cat.categories.tolist())
#g.plot_macrostates(which="terminal", legend_loc="right")

In [ ]:
# Dynamical velocity model -> latent time, a second estimate of each cell's position along the trajectory.
scv.pp.moments(adata_myeloid_clean)

# 2. Run the dynamical model
# This fits the transcriptional dynamics for each gene
scv.tl.recover_dynamics(adata_myeloid_clean)

# 3. Compute velocity using the dynamical model
# 'mode' must be explicitly set to 'dynamical'
scv.tl.velocity(adata_myeloid_clean, mode='dynamical')

# 4. Now you can compute latent time
scv.tl.latent_time(adata_myeloid_clean)

# 5. Finally, plot it
import scanpy as sc
sc.pl.umap(adata_myeloid_clean, color='latent_time', cmap='viridis')
scv.pl.velocity_embedding_stream(adata_myeloid_clean, basis='umap', color='latent_time', title='Kinetic Age and Trajectory')

### Lineage drivers

In [ ]:
# Lineage drivers: genes correlated with the probability of reaching each end state.
# 6. Find genes driving each lineage
lineage_drivers = g.compute_lineage_drivers(return_drivers=True)

# 7. View the top 20 drivers for this terminal state
# >>> CHANGE FOR YOUR DATA: "10" = terminal macrostate name
lineage_drivers.sort_values(by="10_corr", ascending=False).head(20)

In [ ]:
# Volcano-style plot: correlation with fate probability vs significance; top 20 drivers labelled.
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# >>> CHANGE FOR YOUR DATA: "10" = terminal macrostate name
lin = "10"
df = adata_myeloid_clean.varm["terminal_lineage_drivers"].copy()
df = df[[f"{lin}_corr", f"{lin}_qval"]].dropna()
df.columns = ["corr", "qval"]
df["neglog_q"] = -np.log10(df["qval"].clip(lower=1e-300))

top20 = df[df["corr"] > 0].nlargest(20, "neglog_q")

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(df["corr"], df["neglog_q"], s=6, c="lightgrey", alpha=0.5, rasterized=True)
ax.scatter(top20["corr"], top20["neglog_q"], s=30, c="crimson", zorder=3)

ax.set_yscale("log")
# >>> TUNE: axis limits for this lineage (cosmetic)
ax.set_ylim(10, 450)             # crop the empty upper region
ax.set_xlim(0, 0.85)        # crop the empty left tail (negative correlations)

texts = [ax.text(r["corr"], r["neglog_q"], g, fontsize=18, zorder=4)
         for g, r in top20.iterrows()]
adjust_text(texts, ax=ax,
            expand_text=(1.6, 1.8),
            force_text=(0.6, 1.2),
            arrowprops=dict(arrowstyle="-", lw=0.6, color="grey"))

ax.axvline(0, ls="--", lw=0.8, c="black")
ax.set_xlabel(f"Correlation with lineage {lin} fate probability", fontsize=18)
ax.set_ylabel("-log10(q-value)  [log scale]", fontsize=18)
ax.set_title(f"Lineage {lin} drivers", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 driver genes plotted on the UMAP.
# >>> CHANGE FOR YOUR DATA: "10" = terminal macrostate name
g.plot_lineage_drivers(lineage="10", n_genes=20)